# Unit 5 — Unity ML-Agents: Pyramids (curiosity-driven exploration)

**Pass criterion:** threshold −100 → **a correctly pushed model is enough.**

The agent must first **press a button** to spawn a pyramid, then knock it over and reach the **golden brick on top**. The reward is very **sparse** (only at the end), so a **curiosity (RND)** intrinsic reward is used to drive exploration.

No algorithm code: the game is a pre-built Unity environment and PPO is provided by ML-Agents. We prepare the config and run the trainer.

## 1. Setup
ML-Agents only supports **Python 3.10**, while Colab ships a newer Python. We therefore create a separate **Python 3.10.12 environment** (`/content/mlenv`) with `uv` and run all ML-Agents commands from there (~3–5 min).

PyTorch is pinned to **2.2.2** (CPU build): recent PyTorch releases switched `torch.onnx.export` to a new exporter that needs `onnxscript` and breaks ML-Agents' checkpoint export.

In [ ]:
!pip install -q uv
!git clone -q --depth 1 https://github.com/Unity-Technologies/ml-agents /content/ml-agents
!uv venv -q -p 3.10.12 /content/mlenv
!uv pip install -q --python /content/mlenv/bin/python -e /content/ml-agents/ml-agents-envs -e /content/ml-agents/ml-agents
# Newest PyTorch needs onnxscript for ONNX export and breaks ML-Agents checkpoints -> pin the version ML-Agents was tested with
!uv pip install -q --python /content/mlenv/bin/python "torch==2.2.2" "numpy<2" --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

In [ ]:
!/content/mlenv/bin/python --version
!/content/mlenv/bin/mlagents-learn --help | head -3

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ML = "/content/mlenv/bin"       # ML-Agents commands live here

## 2. Hugging Face login (once, at the start)
The model is pushed automatically after training, so we log in first. The cell reads the `HF_TOKEN` Colab secret.

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

## 3. Live training dashboard: TensorBoard
The panel below updates **live** during training (press refresh ⟳ in the top right).
Key chart: **`Environment/Cumulative Reward`** → if it rises, the agent is learning.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/results

---
# Pyramids

## B1. Download the environment

In [ ]:
%%capture
!mkdir -p /content/envs
!wget -q "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" -O /content/envs/Pyramids.zip
!unzip -o -d /content/envs /content/envs/Pyramids.zip
!chmod -R 755 /content/envs/Pyramids/Pyramids

## B2. Training config
We use ML-Agents' own `PyramidsRND.yaml` and only reduce training to **1M steps**.

The new part is the `rnd` reward signal = **curiosity** (Random Network Distillation). There are two networks: a fixed random one and a predictor that tries to imitate it. A large prediction error means "I haven't seen this before" → the agent receives an **intrinsic reward**, which encourages exploration in this sparse-reward game.

In [ ]:
import re
cfg = open("/content/ml-agents/config/ppo/PyramidsRND.yaml").read()
cfg = re.sub(r"max_steps: .*", "max_steps: 1000000", cfg)
open("/content/PyramidsRND.yaml", "w").write(cfg)
print(cfg)

## B3. Training (~40–60 min)
`Mean Reward` starts around −1 (never finds the brick) and climbs **towards +1** as the agent learns to reach it.

In [ ]:
!{ML}/mlagents-learn /content/PyramidsRND.yaml --env=/content/envs/Pyramids/Pyramids --run-id="PyramidsTraining" --results-dir=/content/results --no-graphics --force

## B4. Push to the Hub

In [ ]:
!{ML}/mlagents-push-to-hf --run-id="PyramidsTraining" --local-dir="/content/results/PyramidsTraining" --repo-id="{HF_USERNAME}/ppo-Pyramids" --commit-message="Pyramids PPO + RND agent"

## B5. Watch 🎮
https://huggingface.co/spaces/unity/ML-Agents-Pyramids → select your model (`Zorlu5454/ppo-Pyramids`) and its `.onnx` file → **Watch my agent play**

---
# Check (the same query the certification checker runs)

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")
check_like_certificate("ML-Agents-Pyramids", "ml-agents", -100)
